In [33]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import re

pd.set_option('display.max_columns', None)   # 모든 컬럼 표시
pd.set_option('display.width', None)         # 줄바꿈 없이 전체 폭 사용
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 생략 안 함print(df)

df = pd.read_csv('data/reviews_joined_all_matched.csv')
df.head()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_37348\1111602153.py:23: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/reviews_joined_all_matched.csv')


,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre
0,2139460,199685023,76561198220582271,23,1,15240.0,0.0,15240,NaN,1.748034e+09,turkish,oyunu kurduk eyv oynadık vs zaten türk ü bırak oyuncu bulmak çok zor sadece rus kekolar var bol bol hadi bunları geçtik diyelim oynadık durduk bir güncelleme geldi neymiş yeni senaryo imiş tamam eyv süper güncelleme geliyor bi baktım 70 gb dedik oyun bayağı değişiyor gelişiyor neyse yaptık güncellemeyi baktım bi tek senaryo gelmiş değişen birşey yok tamam dedik eyv. geçen bir kere daha gireyim dedim yeni senaryo gelmiş bi baktım şimdi 64 gb lık bir güncelleme daha .Olum siz manyak mısınız zaten tek tük oyuncu var oynayan birde her senaryoda 70gb güncelleme mi olur mk. mal mısınız sildim sokarım oyununuza. Undawn a devam mk,1752389967,1752389967,False,17,2,0.712445,1,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
1,2139460,199684668,76561198401253542,0,1,6249.0,0.0,834,NaN,1.755746e+09,spanish,me esta fasinando,1752389691,1752389691,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
2,2139460,199683985,76561199435851437,0,1,20522.0,0.0,12327,NaN,1.755124e+09,english,truly an amazing game,1752389204,1752389204,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
3,2139460,199683578,76561198977144059,83,30,22726.0,0.0,4090,NaN,1.755366e+09,english,amazing free to play .,1752388914,1752388914,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
4,2139460,199683575,76561199109403538,0,1,2747.0,0.0,2389,NaN,1.752985e+09,schinese,好,1752388911,1752388911,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"


| 컬럼명(한글) | 컬럼명(영어) | 설명 | 분포(실제 분포) |
|---|---|---|---|
| 게임 ID | appid | Steam 게임 고유 ID | 440 ~ 3,241,660 (mean≈1,239,968) |
| 리뷰 추천 ID | recommendationid | 리뷰 고유 식별자 | 약 1.99e8 ~ 2.15e8 |
| 유저 Steam ID | steamid | 리뷰 작성자 Steam ID | 거의 단일값, 분산 매우 작음 |
| 보유 게임 수 | num_games_owned | 유저가 보유한 전체 게임 수 | 0 ~ 7,706 (median 0, mean 55.9) |
| 작성 리뷰 수 | num_reviews_author | 유저가 작성한 전체 리뷰 수 | 1 ~ 2,542 (median 3) |
| 누적 플레이 타임 | playtime_forever | 해당 게임 총 플레이 시간 | 5 ~ 1,457,369 |
| 최근 2주 플레이 타임 | playtime_last_two_weeks | 최근 2주간 플레이 시간 | 0 ~ 17,144 (75% ≤ 370) |
| 리뷰 시점 플레이 타임 | playtime_at_review | 리뷰 작성 시점 누적 플레이 시간 | 5 ~ 1,396,679 |
| Steam Deck 플레이 타임 | deck_playtime_at_review | 리뷰 시점 Steam Deck 플레이 시간 | 1 ~ 51,724 (표본 적음) |
| 마지막 플레이 시각 | last_played | 마지막 플레이 시점 (Unix) | 1.47e9 ~ 1.77e9 |
| 리뷰 생성 시각 | timestamp_created | 리뷰 최초 작성 시각 (Unix) | 1.75e9 ~ 1.77e9 |
| 리뷰 수정 시각 | timestamp_updated | 리뷰 최종 수정 시각 (Unix) | 생성 시각과 거의 동일 |
| 긍정 추천 여부 | voted_up | 긍정 리뷰 여부 (1=긍정) | mean 0.633 (긍정 약 63%) |
| 도움됨 투표 수 | votes_up | 도움이 됐다고 평가한 수 | 0 ~ 1,511 (대부분 0) |
| 재미있음 투표 수 | votes_funny | 재미있다고 평가한 수 | 0 ~ 438 (대부분 0) |
| 가중 투표 점수 | weighted_vote_score | Steam 내부 도움도 점수 | 0.29 ~ 0.95 (median 0.5) |
| 댓글 수 | comment_count | 리뷰에 달린 댓글 수 | 0 ~ 14 (99% 이상 0) |
| 개발자 응답 시각 | timestamp_dev_responded | 개발자 답변 시각 (Unix) | 존재 데이터 56건 |
| Steam 구매 여부 | steam_purchase | Steam에서 구매했는지 여부 | 없음 |
| 무료 획득 여부 | received_for_free | 무료로 받았는지 여부 | 없음 |
| 얼리액세스 작성 여부 | written_during_early_access | 얼리액세스 중 작성 여부 | 없음 |
| 리뷰 언어 | language | 리뷰 작성 언어 | 없음 |
| 리뷰 텍스트 | review | 리뷰 본문 텍스트 | 없음 |
| 개발자 응답 내용 | developer_response | 개발자 답변 텍스트 | 없음 |
| Steam Deck 주 사용 여부 | primarily_steam_deck | Steam Deck 위주 플레이 여부 | 없음 |
| 중복 appid | appid_1 | appid 중복 컬럼 | appid와 동일 |
| 난수 컬럼 | rnd | 무작위 샘플링용 컬럼 | 0 ~ 0.019 | 

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030656 entries, 0 to 1030655
Data columns (total 28 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   appid                        1030656 non-null  int64  
 1   recommendationid             1030656 non-null  int64  
 2   steamid                      1030656 non-null  int64  
 3   num_games_owned              1030656 non-null  int64  
 4   num_reviews_author           1030656 non-null  int64  
 5   playtime_forever             1030645 non-null  float64
 6   playtime_last_two_weeks      1030645 non-null  float64
 7   playtime_at_review           1030656 non-null  int64  
 8   deck_playtime_at_review      19544 non-null    float64
 9   last_played                  1030645 non-null  float64
 10  language                     1030656 non-null  object 
 11  review                       1027088 non-null  object 
 12  timestamp_created            1030656 non-n

In [3]:
df.isnull().sum()

appid                                0
recommendationid                     0
steamid                              0
num_games_owned                      0
num_reviews_author                   0
playtime_forever                    11
playtime_last_two_weeks             11
playtime_at_review                   0
deck_playtime_at_review        1011112
last_played                         11
language                             0
review                            3568
timestamp_created                    0
timestamp_updated                    0
voted_up                             0
votes_up                             0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
developer_response             1027150
timestamp_dev_responded        1027150
primarily_steam_deck                 0
appid_1                  

In [4]:
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,timestamp_dev_responded,appid_1
count,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030645e+06,1.030645e+06,1.030656e+06,19544.000000,1.030645e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,3.506000e+03,1.030656e+06
mean,1.246007e+06,2.078202e+08,7.656120e+16,5.589029e+01,9.407134e+00,1.421312e+04,4.750142e+02,1.179464e+04,1883.487055,1.762456e+09,1.760941e+09,1.761054e+09,6.485481e-01,1.244169e-01,5.027320e-01,4.645294e-02,1.759162e+09,1.246007e+06
std,8.606580e+05,4.897537e+06,6.086190e+08,1.973416e+02,6.927022e+01,3.357524e+04,1.155320e+03,3.145326e+04,5554.090679,1.173887e+07,4.747926e+06,4.725906e+06,1.345458e+01,3.082606e+00,2.275669e-02,6.260744e-01,4.410209e+06,8.606580e+05
min,4.400000e+02,1.994023e+08,7.656120e+16,0.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00,5.000000e+00,1.000000,1.345532e+09,1.752096e+09,1.752096e+09,0.000000e+00,0.000000e+00,1.352486e-01,0.000000e+00,1.752181e+09,4.400000e+02
25%,5.268700e+05,2.032641e+08,7.656120e+16,0.000000e+00,1.000000e+00,1.440000e+03,0.000000e+00,8.080000e+02,46.000000,1.761593e+09,1.756657e+09,1.756828e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.755116e+09,5.268700e+05
50%,1.172470e+06,2.081491e+08,7.656120e+16,0.000000e+00,3.000000e+00,4.408000e+03,0.000000e+00,2.723000e+03,276.000000,1.765687e+09,1.762017e+09,1.762260e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.759032e+09,1.172470e+06
75%,1.771300e+06,2.122663e+08,7.656120e+16,5.100000e+01,8.000000e+00,1.229500e+04,3.620000e+02,9.001000e+03,1462.000000,1.767210e+09,1.764576e+09,1.764617e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.763067e+09,1.771300e+06
max,3.241660e+06,2.152632e+08,7.656120e+16,3.370000e+04,1.974800e+04,2.457680e+06,3.460800e+04,2.416117e+06,157407.000000,1.767652e+09,1.767650e+09,1.767651e+09,5.763000e+03,1.425000e+03,9.897801e-01,2.580000e+02,1.767650e+09,3.241660e+06


In [5]:
df.columns

Index(['appid', 'recommendationid', 'steamid', 'num_games_owned',
       'num_reviews_author', 'playtime_forever', 'playtime_last_two_weeks',
       'playtime_at_review', 'deck_playtime_at_review', 'last_played',
       'language', 'review', 'timestamp_created', 'timestamp_updated',
       'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score',
       'comment_count', 'steam_purchase', 'received_for_free',
       'written_during_early_access', 'developer_response',
       'timestamp_dev_responded', 'primarily_steam_deck', 'appid_1',
       'game_name', 'genre'],
      dtype='object')

In [6]:
df.count()

appid                          1030656
recommendationid               1030656
steamid                        1030656
num_games_owned                1030656
num_reviews_author             1030656
playtime_forever               1030645
playtime_last_two_weeks        1030645
playtime_at_review             1030656
deck_playtime_at_review          19544
last_played                    1030645
language                       1030656
review                         1027088
timestamp_created              1030656
timestamp_updated              1030656
voted_up                       1030656
votes_up                       1030656
votes_funny                    1030656
weighted_vote_score            1030656
comment_count                  1030656
steam_purchase                 1030656
received_for_free              1030656
written_during_early_access    1030656
developer_response                3506
timestamp_dev_responded           3506
primarily_steam_deck           1030656
appid_1                  

In [7]:

# 1. ROC-AUC 점수 확인을 위한 기본 전처리

df = df.copy()

df["review_dt"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
df = df.dropna(subset=["review_dt"]).copy()

# deck_playtime_at_review 결측 처리 (컬럼 있으면)
if "deck_playtime_at_review" in df.columns:
    df["deck_playtime_at_review"] = df["deck_playtime_at_review"].fillna(0)

# True/False -> 0/1 정리 (LightGBM 용)
bool_cols = [
    "primarily_steam_deck",
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
]
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(int)


In [8]:

# 2. 180일 중 마지막 60일 제외

# 마지막 60일은 미래 정보에 해당하므로 라벨 생성에서 제외
# 과거 120일 구간만 사용해 학습용 라벨을 안정적으로 정의
END_DATE = df["review_dt"].max()
START_DATE = END_DATE - pd.Timedelta(days=180)
LABEL_CUTOFF = END_DATE - pd.Timedelta(days=60)

df_label = df[df["review_dt"] <= LABEL_CUTOFF].copy()

### 해당 프로젝트는 리뷰 시점을 기준으로 이탈을 예측하는 문제로 정의했습니다.
### 따라서 리뷰 이후의 행동만을 기준으로 타겟을 정의하여 예측 시점과 라벨 시점이 섞이는 데이터 누수를 방지했습니다.
### 데이터 수집 오류로 인해 리뷰 이전 시점의 last_played가 존재하는 경우는
### 정상적인 복귀로 해석할 수 없기 때문에 이탈로 보수적으로 처리했습니다.

In [9]:
# 3. churn 생성
df_label["last_played_dt"] = pd.to_datetime(df_label["last_played"], unit="s", errors="coerce")

# last_played가 Null이면 복귀 관측 안돼서 churn=1 처리
df_label["churn"] = (
    df_label["last_played_dt"].isna() | (df_label["last_played_dt"] <= (df_label["review_dt"] + pd.Timedelta(days=30)))
).astype(int)

In [10]:

# 4. 피처 선택 (도메인 기반 필터)
features = [
    "num_games_owned",
    "num_reviews_author",
    "deck_playtime_at_review",
    "voted_up",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "review_length"
]

# 존재하는 컬럼만 사용 (실행 에러 방지)
features = [c for c in features if c in df_label.columns]

# 숫자형 강제 (문자 섞이면 터짐)
for c in features:
    df_label[c] = pd.to_numeric(df_label[c], errors="coerce")

# 결측은 0으로 채움
X = df_label[features].fillna(0)
y = df_label["churn"].astype(int)


In [11]:
# 5. 시간 기준 Train/Valid Split (마지막 30일을 valid)
split_date = LABEL_CUTOFF - pd.Timedelta(days=30)

train_mask = df_label["review_dt"] <= split_date
valid_mask = df_label["review_dt"] > split_date

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]

print("Rows:", len(df_label), "| Train:", len(X_train), "| Valid:", len(X_valid))
print("Churn rate train:", round(y_train.mean(), 4), "| valid:", round(y_valid.mean(), 4))
print("Features used:", features)

# valid에 한 클래스만 있으면 AUC 계산이 안 됨
if y_valid.nunique() < 2:
    raise ValueError(f"Valid set에 클래스가 1개뿐입니다. (unique={y_valid.unique()}) split_date를 조정하거나 기간을 늘려야 합니다.")


Rows: 535135 | Train: 414585 | Valid: 120550
Churn rate train: 0.3508 | valid: 0.4337
Features used: ['num_games_owned', 'num_reviews_author', 'deck_playtime_at_review', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access']


In [12]:
# 6. LightGBM 학습 + ROC-AUC
model = LGBMClassifier(
    objective="binary",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc"
)

pred = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, pred)
print(f"Validation ROC-AUC: {auc:.4f}")

[LightGBM] [Info] Number of positive: 145442, number of negative: 269143
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009766 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1256
[LightGBM] [Info] Number of data points in the train set: 414585, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.350813 -> initscore=-0.615465
[LightGBM] [Info] Start training from score -0.615465
Validation ROC-AUC: 0.6252


## AUC-ROC 점수가 유의미한 점수라고 판단 해당 데이터셋 선택 최종 확정

## EDA

In [13]:
df = pd.read_csv('data/reviews_joined_all_matched.csv')

C:\Users\Playdata\AppData\Local\Temp\ipykernel_37348\3942676616.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/reviews_joined_all_matched.csv')


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030656 entries, 0 to 1030655
Data columns (total 28 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   appid                        1030656 non-null  int64  
 1   recommendationid             1030656 non-null  int64  
 2   steamid                      1030656 non-null  int64  
 3   num_games_owned              1030656 non-null  int64  
 4   num_reviews_author           1030656 non-null  int64  
 5   playtime_forever             1030645 non-null  float64
 6   playtime_last_two_weeks      1030645 non-null  float64
 7   playtime_at_review           1030656 non-null  int64  
 8   deck_playtime_at_review      19544 non-null    float64
 9   last_played                  1030645 non-null  float64
 10  language                     1030656 non-null  object 
 11  review                       1027088 non-null  object 
 12  timestamp_created            1030656 non-n

In [15]:
df.isnull().sum()

appid                                0
recommendationid                     0
steamid                              0
num_games_owned                      0
num_reviews_author                   0
playtime_forever                    11
playtime_last_two_weeks             11
playtime_at_review                   0
deck_playtime_at_review        1011112
last_played                         11
language                             0
review                            3568
timestamp_created                    0
timestamp_updated                    0
voted_up                             0
votes_up                             0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
developer_response             1027150
timestamp_dev_responded        1027150
primarily_steam_deck                 0
appid_1                  

In [16]:
# 결측치 제거
df.drop(columns=['developer_response','timestamp_dev_responded'],inplace=True)
df = df.dropna(subset=['review','playtime_last_two_weeks','playtime_forever','last_played'])
df.isnull().sum()

appid                                0
recommendationid                     0
steamid                              0
num_games_owned                      0
num_reviews_author                   0
playtime_forever                     0
playtime_last_two_weeks              0
playtime_at_review                   0
deck_playtime_at_review        1007593
last_played                          0
language                             0
review                               0
timestamp_created                    0
timestamp_updated                    0
voted_up                             0
votes_up                             0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
primarily_steam_deck                 0
appid_1                              0
game_name                            0
genre                    

In [17]:
# 이상치
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,appid_1
count,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,19484.000000,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06
mean,1.246218e+06,2.078205e+08,7.656120e+16,5.591921e+01,9.398358e+00,1.422662e+04,4.750545e+02,1.180767e+04,1887.244662,1.762453e+09,1.760942e+09,1.761054e+09,6.506211e-01,1.248115e-01,5.027439e-01,4.661189e-02,1.246218e+06
std,8.609537e+05,4.896923e+06,6.085837e+08,1.974617e+02,6.935847e+01,3.360347e+04,1.155415e+03,3.148088e+04,5561.559578,1.175115e+07,4.747388e+06,4.725320e+06,1.347795e+01,3.087957e+00,2.278943e-02,6.271562e-01,8.609537e+05
min,4.400000e+02,1.994023e+08,7.656120e+16,0.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00,5.000000e+00,1.000000,1.345532e+09,1.752096e+09,1.752096e+09,0.000000e+00,0.000000e+00,1.352486e-01,0.000000e+00,4.400000e+02
25%,5.268700e+05,2.032650e+08,7.656120e+16,0.000000e+00,1.000000e+00,1.440000e+03,0.000000e+00,8.080000e+02,46.000000,1.761592e+09,1.756658e+09,1.756830e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,5.268700e+05
50%,1.172470e+06,2.081504e+08,7.656120e+16,0.000000e+00,3.000000e+00,4.411000e+03,0.000000e+00,2.727000e+03,277.000000,1.765687e+09,1.762018e+09,1.762262e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.172470e+06
75%,1.771300e+06,2.122644e+08,7.656120e+16,5.100000e+01,8.000000e+00,1.230900e+04,3.620000e+02,9.012000e+03,1465.000000,1.767210e+09,1.764574e+09,1.764615e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.771300e+06
max,3.241660e+06,2.152632e+08,7.656120e+16,3.370000e+04,1.974800e+04,2.457680e+06,3.460800e+04,2.416117e+06,157407.000000,1.767652e+09,1.767650e+09,1.767651e+09,5.763000e+03,1.425000e+03,9.897801e-01,2.580000e+02,3.241660e+06


In [18]:
# 플레 시간 관련 컬럼들이 말도 안되는 초헤비 유저 > 봇사용 의심 + 영향력 
# 1. 상위 99.9% 캡 
cap = df["playtime_forever"].quantile(0.999)

# 2. 캡 + log1p 처리해서 바로 덮어쓰기
df["playtime_forever"] = np.log1p(
    df["playtime_forever"].clip(upper=cap)
)

# 플레이 시간은 누수가 있어서 피쳐로는 사용 안하지만 플레이시간을 제거함으로써 다른 시간 이상치를 같이 처리 해주었다

In [19]:
# num_games_owned
# median: 0
# 75%: 51
# max: 33,700
# std: 197
cap = df["num_games_owned"].quantile(0.999)
df["num_games_owned"] = np.log1p(df["num_games_owned"].clip(upper=cap))

In [20]:
# num_reviews_author
# median: 3
# 75%: 8
# max: 19,748
cap = df["num_reviews_author"].quantile(0.999)
df["num_reviews_author"] = np.log1p(df["num_reviews_author"].clip(upper=cap))

In [21]:
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,appid_1
count,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,19484.000000,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06,1.027077e+06
mean,1.246218e+06,2.078205e+08,7.656120e+16,1.862805e+00,1.662801e+00,8.350030e+00,4.750545e+02,1.180767e+04,1887.244662,1.762453e+09,1.760942e+09,1.761054e+09,6.506211e-01,1.248115e-01,5.027439e-01,4.661189e-02,1.246218e+06
std,8.609537e+05,4.896923e+06,6.085837e+08,2.180613e+00,9.557001e-01,1.630759e+00,1.155415e+03,3.148088e+04,5561.559578,1.175115e+07,4.747388e+06,4.725320e+06,1.347795e+01,3.087957e+00,2.278943e-02,6.271562e-01,8.609537e+05
min,4.400000e+02,1.994023e+08,7.656120e+16,0.000000e+00,6.931472e-01,1.791759e+00,0.000000e+00,5.000000e+00,1.000000,1.345532e+09,1.752096e+09,1.752096e+09,0.000000e+00,0.000000e+00,1.352486e-01,0.000000e+00,4.400000e+02
25%,5.268700e+05,2.032650e+08,7.656120e+16,0.000000e+00,6.931472e-01,7.273093e+00,0.000000e+00,8.080000e+02,46.000000,1.761592e+09,1.756658e+09,1.756830e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,5.268700e+05
50%,1.172470e+06,2.081504e+08,7.656120e+16,0.000000e+00,1.386294e+00,8.392083e+00,0.000000e+00,2.727000e+03,277.000000,1.765687e+09,1.762018e+09,1.762262e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.172470e+06
75%,1.771300e+06,2.122644e+08,7.656120e+16,3.951244e+00,2.197225e+00,9.418167e+00,3.620000e+02,9.012000e+03,1465.000000,1.767210e+09,1.764574e+09,1.764615e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.771300e+06
max,3.241660e+06,2.152632e+08,7.656120e+16,7.674582e+00,5.676754e+00,1.285269e+01,3.460800e+04,2.416117e+06,157407.000000,1.767652e+09,1.767650e+09,1.767651e+09,5.763000e+03,1.425000e+03,9.897801e-01,2.580000e+02,3.241660e+06


In [22]:
# plt.figure(figsize=(30,5))

# sns.boxenplot(df)

In [23]:
# 장르에 따른 분류
ending_genre = [
  "Visual Novel",
  "Interactive Fiction",
  "Walking Simulator",
  "Story Rich",
  "Adventure",
  "Puzzle",
  "Horror",
  "Mystery",
  "Psychological Horror",
  "Narrative"
]

df["is_ending_genre"] = df["genre"].apply(
    lambda g: int(any(x in g for x in ending_genre))
)

In [24]:
# 1) 언어별 키워드 사전
# - phrases: 문장/구문(부분일치 OK)
# - words: 단어성 키워드(라틴권은 단어경계 \b 적용)
# - neg: 부정 구문(걸리면 good=0으로 처리)
# - boundary: words에 \b를 붙일지 여부 (중국어/일본어/태국어/한국어는 보통 False)
LEXICON = {
    # English
    "english": {
        "phrases": [
            r"highly recommend(?:ed)?",
            r"definitely recommend",
            r"worth (?:buying|it|the money|the time)",
            r"great game",
            r"amazing game",
            r"awesome game",
            r"best game(?:s)?",
        ],
        "words": [
            r"awesome", r"amazing", r"great", r"excellent", r"fantastic", r"incredible",
            r"masterpiece", r"perfect", r"love", r"fun", r"enjoy", r"recommend", r"worth",
        ],
        "neg": [
            r"not\s+good", r"not\s+great", r"not\s+worth",
            r"(?:do\s*not|don't|dont)\s+recommend",
            r"(?:do\s*not|don't|dont)\s+buy",
            r"can't\s+recommend|cant\s+recommend",
            r"avoid\b", r"refund\b",
        ],
        "boundary": True,
    },

    # Spanish (Spain) + LatAm는 같이 처리
    "spanish": {
        "phrases": [r"muy bueno", r"vale la pena", r"lo recomiendo", r"recomendad[oa]"],
        "words": [r"genial", r"excelente", r"buen[oa]", r"incre[ií]ble", r"recomiendo", r"recomendar"],
        "neg": [r"no\s+recomiendo", r"no\s+vale\s+la\s+pena", r"no\s+es\s+buen[oa]", r"no\s+merece\s+la\s+pena", r"no\s+compr(?:es|ar)"],
        "boundary": True,
    },
    "latam": {  # 라틴아메리카 스페인어
        "phrases": [r"muy bueno", r"vale la pena", r"lo recomiendo", r"recomendad[oa]"],
        "words": [r"genial", r"excelente", r"buen[oa]", r"incre[ií]ble", r"recomiendo", r"recomendar"],
        "neg": [r"no\s+recomiendo", r"no\s+vale\s+la\s+pena", r"no\s+es\s+buen[oa]", r"no\s+merece\s+la\s+pena", r"no\s+compr(?:es|ar)"],
        "boundary": True,
    },

    # Portuguese (PT / BR)
    "portuguese": {
        "phrases": [r"vale a pena", r"recomendo", r"muito bom", r"jogo (?:muito )?bom"],
        "words": [r"ótimo", r"excelente", r"incr[ií]vel", r"perfeito", r"divertido", r"recomendar"],
        "neg": [r"não\s+recomendo", r"nao\s+recomendo", r"não\s+vale\s+a\s+pena", r"nao\s+vale\s+a\s+pena", r"não\s+é\s+bom", r"nao\s+e\s+bom", r"não\s+compr(?:e|ar)", r"nao\s+compr(?:e|ar)"],
        "boundary": True,
    },
    "brazilian": {  # 브라질 포르투갈어
        "phrases": [r"vale a pena", r"recomendo", r"muito bom", r"jogo (?:muito )?bom"],
        "words": [r"ótimo", r"excelente", r"incr[ií]vel", r"perfeito", r"divertido", r"recomendar"],
        "neg": [r"não\s+recomendo", r"nao\s+recomendo", r"não\s+vale\s+a\s+pena", r"nao\s+vale\s+a\s+pena", r"não\s+é\s+bom", r"nao\s+e\s+bom", r"não\s+compr(?:e|ar)", r"nao\s+compr(?:e|ar)"],
        "boundary": True,
    },

    # German
    "german": {
        "phrases": [r"sehr gut", r"klare(?:s)? empfehlung", r"lohnt sich", r"absolut empfehl"],
        "words": [r"genial", r"toll", r"super", r"großartig", r"exzellent", r"empfehle", r"empfehlenswert"],
        "neg": [r"nicht\s+empfehl", r"lohnt\s+sich\s+nicht", r"nicht\s+gut", r"kau(?:f|ft)\s+nicht", r"kein\s+kauf"],
        "boundary": True,
    },

    # French
    "french": {
        "phrases": [r"je recommande", r"vaut le coup", r"tr[eè]s bon", r"excellent jeu"],
        "words": [r"g[eé]nial", r"excellent", r"super", r"incroyable", r"parfait", r"recommande"],
        "neg": [r"je\s+ne\s+recommande\s+pas", r"ne\s+vaut\s+pas\s+le\s+coup", r"pas\s+bon", r"n['’]achetez\s+pas", r"n['’]ach[eè]te\s+pas"],
        "boundary": True,
    },

    # Italian
    "italian": {
        "phrases": [r"lo consiglio", r"vale la pena", r"molto bello", r"gioco (?:molto )?bello"],
        "words": [r"fantastico", r"ottimo", r"eccellente", r"stupendo", r"divertente", r"consiglio", r"consigliare"],
        "neg": [r"non\s+lo\s+consiglio", r"non\s+vale\s+la\s+pena", r"non\s+[eè]\s+bello", r"non\s+compr(?:are|atelo)"],
        "boundary": True,
    },

    # Dutch
    "dutch": {
        "phrases": [r"zeker aanraden", r"de moeite waard", r"heel goed", r"geweldig spel"],
        "words": [r"geweldig", r"fantastisch", r"super", r"leuk", r"aanraden", r"aanbevelen", r"waarde"],
        "neg": [r"niet\s+aanrad", r"niet\s+de\s+moeite\s+waard", r"niet\s+goed", r"koop\s+niet"],
        "boundary": True,
    },

    # Swedish / Norwegian / Danish / Finnish
    "swedish": {
        "phrases": [r"rekommenderar", r"värt det", r"jättebra", r"riktigt bra"],
        "words": [r"fantastisk", r"grym", r"suverän", r"toppen", r"kul", r"rekommendera", r"värd"],
        "neg": [r"rekommenderar\s+inte", r"inte\s+värt", r"inte\s+bra", r"köp\s+inte"],
        "boundary": True,
    },
    "norwegian": {
        "phrases": [r"anbefaler", r"verdt det", r"kjempebra", r"veldig bra"],
        "words": [r"fantastisk", r"råbra", r"suveren", r"gøy", r"anbefale", r"verdt"],
        "neg": [r"anbefaler\s+ikke", r"ikke\s+verdt", r"ikke\s+bra", r"ikke\s+kjøp"],
        "boundary": True,
    },
    "danish": {
        "phrases": [r"anbefaler", r"v[æa]rd at", r"mega god", r"rigtig god"],
        "words": [r"fantastisk", r"fremragende", r"super", r"sjov", r"anbefale", r"v[æa]rd"],
        "neg": [r"anbefaler\s+ikke", r"ikke\s+v[æa]rd", r"ikke\s+god", r"k[oø]b\s+ikke"],
        "boundary": True,
    },
    "finnish": {
        "phrases": [r"suosittelen", r"todella hyv[äa]", r"sen arvoinen", r"hyv[äa] peli"],
        "words": [r"loistava", r"mahtava", r"erinomainen", r"hauska", r"suositella", r"arvoinen"],
        "neg": [r"en\s+suosittele", r"ei\s+kannata", r"ei\s+hyv[äa]", r"älä\s+osta"],
        "boundary": True,
    },

    # Polish / Czech / Romanian / Hungarian / Bulgarian / Greek / Ukrainian / Russian / Turkish
    "polish": {
        "phrases": [r"polecam", r"warto", r"świetna gra", r"bardzo dobra"],
        "words": [r"świetn[aey]", r"super", r"rewelacyjna", r"doskonała", r"polecić", r"warto"],
        "neg": [r"nie\s+polecam", r"nie\s+warto", r"nie\s+jest\s+dobr", r"nie\s+kupuj"],
        "boundary": True,
    },
    "czech": {
        "phrases": [r"doporu[čc]uji", r"stoj[ií]\s+za\s+to", r"skv[ěe]l[aá]", r"v[ýy]born[aá]"],
        "words": [r"super", r"skv[ěe]l", r"v[ýy]born", r"bav[ií]", r"doporu[čc]it"],
        "neg": [r"nedoporu[čc]uji", r"nestoj[ií]\s+za\s+to", r"nen[ií]\s+dobr", r"nekupuj"],
        "boundary": True,
    },
    "romanian": {
        "phrases": [r"recomand", r"merit[ăa]", r"foarte bun", r"joc (?:foarte )?bun"],
        "words": [r"excelent", r"minunat", r"super", r"recomanda", r"merit"],
        "neg": [r"nu\s+recomand", r"nu\s+merit[ăa]", r"nu\s+e\s+bun", r"nu\s+cump[ăa]ra"],
        "boundary": True,
    },
    "hungarian": {
        "phrases": [r"aj[aá]nlom", r"meg[eé]ri", r"nagyon j[oó]", r"szuper j[aá]t[eé]k"],
        "words": [r"szuper", r"fantasztikus", r"kiv[aá]l[oó]", r"nagyon", r"aj[aá]nlani", r"meg[eé]r"],
        "neg": [r"nem\s+aj[aá]nlom", r"nem\s+[eé]ri\s+meg", r"nem\s+j[oó]", r"ne\s+vedd\s+meg"],
        "boundary": True,
    },
    "bulgarian": {
        "phrases": [r"препоръч", r"много добра", r"страхотна", r"заслужава си"],
        "words": [r"страхот", r"отлич", r"супер", r"препоръч", r"шедьовър"],
        "neg": [r"не\s+препоръч", r"не\s+си\s+струва", r"не\s+е\s+доб", r"не\s+купувай"],
        "boundary": False,  # кир릴은 \b가 애매해서 단순부분일치로
    },
    "greek": {
        "phrases": [r"το\s+προτείν", r"αξίζει", r"πολύ\s+καλ", r"εξαιρετικ"],
        "words": [r"τέλει", r"φοβε", r"εξαιρετικ", r"καταπληκτικ", r"προτείν", r"αξίζ"],
        "neg": [r"δεν\s+προτείν", r"δεν\s+αξίζ", r"δεν\s+είναι\s+καλ", r"μην\s+αγοράσ"],
        "boundary": False,
    },
    "ukrainian": {
        "phrases": [r"рекоменд", r"дуже\s+хорош", r"варто", r"чудов"],
        "words": [r"відмін", r"класн", r"шедевр", r"рекоменд", r"варто"],
        "neg": [r"не\s+рекоменд", r"не\s+варто", r"не\s+хорош", r"не\s+купуй"],
        "boundary": False,
    },
    "russian": {
        "phrases": [r"рекоменд", r"очень\s+хорош", r"стоит", r"шедевр"],
        "words": [r"отлич", r"классн", r"супер", r"шедевр", r"рекоменд", r"стоит"],
        "neg": [r"не\s+рекоменд", r"не\s+стоит", r"плох", r"не\s+покупай", r"не\s+берите"],
        "boundary": False,
    },
    "turkish": {
        "phrases": [r"kesinlikle tavsiye", r"tavsiye ederim", r"çok iyi", r"mükemmel", r"harika"],
        "words": [r"güzel", r"mükemmel", r"harika", r"şahane", r"tavsiye", r"değer"],
        "neg": [r"tavsiye etmem", r"tavsiye etmiyorum", r"iyi değil", r"alma", r"almayın", r"değmez"],
        "boundary": True,
    },

    # Korean / Japanese / Chinese / Arabic / Thai / Vietnamese / Indonesian
    "koreana": {
        "phrases": [r"강추", r"완전 추천", r"강력 추천", r"갓겜", r"명작", r"존잼", r"개꿀잼", r"재밌", r"재미있"],
        "words": [r"추천", r"최고", r"꿀잼", r"재미", r"좋다", r"훌륭", r"완벽", r"감동"],
        "neg": [r"비추", r"추천\s*안", r"추천\s*하지", r"재미없", r"별로", r"최악", r"사지\s*마", r"사지마", r"환불"],
        "boundary": False,
    },
    "japanese": {
        "phrases": [r"おすすめ", r"オススメ", r"最高", r"神ゲー", r"買う価値", r"面白い", r"楽しい"],
        "words": [r"おすすめ", r"最高", r"神", r"面白", r"楽しい", r"良い", r"素晴らしい"],
        "neg": [r"おすすめしない", r"買わない方が", r"つまらない", r"面白くない", r"最悪", r"返品"],
        "boundary": False,
    },
    "schinese": {
        "phrases": [r"强烈推荐", r"非常推荐", r"值得买", r"值得入", r"很值得", r"很好玩", r"神作", r"精品"],
        "words": [r"推荐", r"值得", r"好玩", r"很好", r"优秀", r"完美", r"喜欢"],
        "neg": [r"不推荐", r"不值得", r"不好玩", r"垃圾", r"别买", r"千万别买", r"退款"],
        "boundary": False,
    },
    "tchinese": {
        "phrases": [r"強烈推薦", r"非常推薦", r"值得買", r"值得入", r"很值得", r"很好玩", r"神作", r"精品"],
        "words": [r"推薦", r"值得", r"好玩", r"很好", r"優秀", r"完美", r"喜歡"],
        "neg": [r"不推薦", r"不值得", r"不好玩", r"垃圾", r"別買", r"千萬別買", r"退款"],
        "boundary": False,
    },
    "arabic": {
        "phrases": [r"أنصح", r"ممتاز", r"رائع", r"يستحق", r"لعبة رائعة", r"ممتعة"],
        "words": [r"ممتاز", r"رائع", r"جميل", r"ممتع", r"يستحق", r"أنصح"],
        "neg": [r"لا\s+أنصح", r"لا\s+يستحق", r"سيئ", r"لا\s+تشتري", r"استرجاع"],
        "boundary": False,
    },
    "thai": {
        "phrases": [r"แนะนำ", r"ดีมาก", r"สุดยอด", r"คุ้มค่า", r"สนุกมาก", r"โคตรสนุก"],
        "words": [r"แนะนำ", r"ดี", r"สนุก", r"สุดยอด", r"คุ้ม", r"ชอบ"],
        "neg": [r"ไม่แนะนำ", r"ไม่คุ้ม", r"ไม่ดี", r"แย่", r"อย่าซื้อ", r"ขอคืนเงิน"],
        "boundary": False,
    },
    "vietnamese": {
        "phrases": [r"rất hay", r"tuyệt vời", r"đáng mua", r"đáng tiền", r"nên mua", r"khuyên dùng"],
        "words": [r"hay", r"tuyệt", r"xuất sắc", r"đáng", r"thích", r"khuyên", r"nên"],
        "neg": [r"không\s+khuyên", r"không\s+đáng", r"đừng\s+mua", r"tệ", r"chán", r"hoàn tiền"],
        "boundary": True,
    },
    "indonesian": {
        "phrases": [r"sangat bagus", r"rekomendasi", r"worth it", r"layak dibeli", r"seru banget"],
        "words": [r"bagus", r"keren", r"mantap", r"seru", r"rekomend", r"layak"],
        "neg": [r"tidak\s+rekomend", r"jangan\s+beli", r"tidak\s+layak", r"jelek", r"buruk", r"refund"],
        "boundary": True,
    },
}

# 없는 언어는 english로 fallback
DEFAULT_LANG = "english"


# 2) 정규식 빌더
def _compile_lexicon(cfg):
    # 언어 별 \b(단어경계)를 쓸지 말지 결정
    # 예를 들어 라틴 알파벳 계열은 단어 \b를 써야 bad가 badly 단어에 붙어서 오탐나는거 방지
    boundary = cfg.get("boundary", True)

    parts_good = []
    for p in cfg.get("phrases", []):
        parts_good.append(f"(?:{p})") # ?:...를 사용하는 이유: 정규식에서 ...은 캡처를 만들고, 매칭 결과에 그룹이 저장.
                                      # ?:...은 그룹화는 하지만 캡처는 안함
    for w in cfg.get("words", []):    # 저희는 or(|)로 묶어서 패턴을 합치는 목적이라서 캡처가 필요없습니다
        if boundary:
            parts_good.append(rf"\b{w}\b")
        else:
            parts_good.append(f"(?:{w})")

    good_pat = "|".join(parts_good) if parts_good else r"$^"  #  r"$^"구문은 비어있을 때 매칭 안 되게 막아주는 역할
    good_re = re.compile(good_pat, flags=re.UNICODE) # 비라틴 문자(한글/중국어/키릴 등) 섞여도 정규식 엔진이 유니코드로 처리

    neg_parts = [f"(?:{p})" for p in cfg.get("neg", [])]
    neg_pat = "|".join(neg_parts) if neg_parts else r"$^"
    neg_re = re.compile(neg_pat, flags=re.UNICODE)

    return good_re, neg_re


# 미리 컴파일 -> 언어별로 딱 한 번만 컴파일 해두고 재사용하는 방식
# 데이터가 100만행 단위일 땐, 행마다 컴파일 성능이 크게 떨어지기 때문
_COMPILED = {}
for lang, cfg in LEXICON.items():
    _COMPILED[lang] = _compile_lexicon(cfg)
_COMPILED[DEFAULT_LANG] = _COMPILED.get(DEFAULT_LANG, _compile_lexicon(LEXICON["english"]))


# 3) 멀티언어 good_review 생성 함수
def add_good_flag_multilang(df, text_col="review", lang_col="language"):
    out = df.copy() # 원본 df 망가뜨리지 않기 위해 복사본에서 작업

    # casefold: lower보다 더 강한 소문자화(터키어 등)
    text = out[text_col].fillna("").astype(str).str.casefold()
    lang = out[lang_col].fillna(DEFAULT_LANG).astype(str)

    good_hit = pd.Series(False, index=out.index) # False 선언: 언어 없는 행이나, 처리 하지 못하는 언어는 안전하게 기본값으로 남기기
    neg_hit  = pd.Series(False, index=out.index)

    # 언어별 반복문을 행 단위가 아니라 언어 단위로 돌아야함
    # 예를 들어 100만행을 100만번 도는 방식은 너무 느리기 때문에 언어 종류 수만큼만 반복실행 (예:30개)
    for l in lang.unique():
        mask = (lang == l)
        # 리뷰로 남겨진 언어가 language쪽에 존재하지 않는다면 default로 영어 정규식 사용
        # 그래서 언어 코드가 예상과 달라도 에러 발생X
        good_re, neg_re = _COMPILED.get(l, _COMPILED[DEFAULT_LANG])

        # 각 리뷰에서 good_re 패턴이 한번이라도 매칭되면 True
        good_hit.loc[mask] = text.loc[mask].str.contains(good_re, regex=True)
        neg_hit.loc[mask]  = text.loc[mask].str.contains(neg_re,  regex=True)

    # 최종 라벨 생성
    # good 조건을 만족하더라도 neg 조건이 잡히면 good리뷰로 보지 않고 탈락시키는 구문
    out["good_review"] = (good_hit & (~neg_hit)).astype(int)
    return out


# 4) 적용
# temp 데이터프레임에 대해 생성
# 결과로 good_review가 추가된 df 반환
# 분포 확인은 라벨이 너무 한쪽으로 쏠리는 지 출력으로 확인
df = add_good_flag_multilang(df, text_col="review", lang_col="language")
print(df["good_review"].value_counts())


good_review
0    750649
1    276428
Name: count, dtype: int64


In [25]:
# 1. ROC-AUC 점수 확인을 위한 기본 전처리

df["review_dt"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
df = df.dropna(subset=["review_dt"]).copy()

# deck_playtime_at_review 결측 처리 (컬럼 있으면)
if "deck_playtime_at_review" in df.columns:
    df["deck_playtime_at_review"] = df["deck_playtime_at_review"].fillna(0)

# True/False -> 0/1 정리 (LightGBM 용)
bool_cols = [
    "primarily_steam_deck",
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
]
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(int)


In [26]:

# 2. 180일 중 마지막 60일 제외

# 마지막 60일은 미래 정보에 해당하므로 라벨 생성에서 제외
# 과거 120일 구간만 사용해 학습용 라벨을 안정적으로 정의
END_DATE = df["review_dt"].max()
START_DATE = END_DATE - pd.Timedelta(days=180)
LABEL_CUTOFF = END_DATE - pd.Timedelta(days=60)

df_label = df[df["review_dt"] <= LABEL_CUTOFF].copy()
# 3. churn 생성
df_label["last_played_dt"] = pd.to_datetime(df_label["last_played"], unit="s", errors="coerce")

# last_played가 Null이면 복귀 관측 안돼서 churn=1 처리
df_label["churn"] = (
    df_label["last_played_dt"].isna() | (df_label["last_played_dt"] <= (df_label["review_dt"] + pd.Timedelta(days=30)))
).astype(int)

In [27]:
# 4. 피처 선택 (도메인 기반 필터)
features = [
    "num_games_owned",
    "num_reviews_author",
    "deck_playtime_at_review",
    "voted_up",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "review_length",
    'is_ending_genre',
    'playtime_at_review',
    'good_review'
]

# 존재하는 컬럼만 사용 (실행 에러 방지)
features = [c for c in features if c in df_label.columns]

# 숫자형 강제 (문자 섞이면 터짐)
for c in features:
    df_label[c] = pd.to_numeric(df_label[c], errors="coerce")

# 결측은 0으로 채움
X = df_label[features].fillna(0)
y = df_label["churn"].astype(int)

In [28]:
pd.crosstab(df["voted_up"], df["good_review"], normalize="index")

good_review,0,1
voted_up,,
0,0.887570,0.112430
1,0.705805,0.294195


In [29]:
# 5. 시간 기준 Train/test Split (마지막 30일을 test)
split_date = LABEL_CUTOFF - pd.Timedelta(days=30)

train_mask = df_label["review_dt"] <= split_date
test_mask = df_label["review_dt"] > split_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Rows:", len(df_label), "| Train:", len(X_train), "| test:", len(X_test))
print("Churn rate train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))
print("Features used:", features)

# test에 한 클래스만 있으면 AUC 계산이 안 됨
if y_test.nunique() < 2:
    raise ValueError(f"test set에 클래스가 1개뿐입니다. (unique={y_test.unique()}) split_date를 조정하거나 기간을 늘려야 합니다.")


Rows: 533203 | Train: 413078 | test: 120125
Churn rate train: 0.3508 | test: 0.4337
Features used: ['num_games_owned', 'num_reviews_author', 'deck_playtime_at_review', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'is_ending_genre', 'playtime_at_review', 'good_review']


### 이탈 데이터는 변수 간 비선형 상관관계가 강해, 이를 잘 포착하는 LightGBM을 사용해 초기 feature 중요도를 산출

In [36]:
# 6. LightGBM 학습 + precision, recall, f1
lgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=800,               # 트리 개수
    learning_rate=0.05,             # 학습률
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=7777               # 재현성 확보 (7777)
)

lgbm_model.fit(X_train, y_train)

lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]

# 임계값 범위 설정
thresholds = np.arange(0.05, 0.51, 0.05)
rows = []

# 임계값 별 계산
for t in thresholds:
    y_pred = (lgbm_prob >= t).astype(int)  # 임계값 기준으로  이진 분류

    print(f"threshold: {t}")
    print(f"precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"f1: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print("--------------------------------------------------------------")

[LightGBM] [Info] Number of positive: 144897, number of negative: 268181
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011731 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1468
[LightGBM] [Info] Number of data points in the train set: 413078, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.350774 -> initscore=-0.615639
[LightGBM] [Info] Start training from score -0.615639
threshold: 0.05
precision: 0.4372
recall: 0.9982
f1: 0.6080
--------------------------------------------------------------
threshold: 0.1
precision: 0.4563
recall: 0.9814
f1: 0.6230
--------------------------------------------------------------
threshold: 0.15000000000000002
precision: 0.4749
recall: 0.9497
f1: 0.6332
--------------------------------------------------------------
threshold: 0.2
precision: 0.4948
recall: 0.9056
f1: 0.63

In [35]:
# 6. xgboost 학습 + precision, recall, f1
xgb_model = XGBClassifier(
    objective="binary:logistic",  # 이진 분류
    n_estimators=800,             # 트리 개수
    learning_rate=0.05,           # 학습률
    max_depth=6,                  # 트리 깊이 (num_leaves 대체)
    subsample=0.8,                # row sampling
    colsample_bytree=0.8,         # feature sampling
    eval_metric="logloss",        # 경고 제거 + 안정성
    random_state=7777,
    n_jobs=-1                     # 병렬 처리
)
xgb_model.fit(X_train, y_train)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# 임계값 범위 설정
thresholds = np.arange(0.05, 0.51, 0.05)
rows = []

# 임계값 별 계산
for t in thresholds:
    y_pred = (xgb_prob >= t).astype(int)  # 임계값 기준으로  이진 분류

    print(f"threshold: {t}")
    print(f"precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"f1: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print("--------------------------------------------------------------")

threshold: 0.05
precision: 0.4377
recall: 0.9983
f1: 0.6086
--------------------------------------------------------------
threshold: 0.1
precision: 0.4569
recall: 0.9814
f1: 0.6235
--------------------------------------------------------------
threshold: 0.15000000000000002
precision: 0.4748
recall: 0.9499
f1: 0.6331
--------------------------------------------------------------
threshold: 0.2
precision: 0.4950
recall: 0.9052
f1: 0.6400
--------------------------------------------------------------
threshold: 0.25
precision: 0.5254
recall: 0.8233
f1: 0.6415
--------------------------------------------------------------
threshold: 0.3
precision: 0.5578
recall: 0.7170
f1: 0.6275
--------------------------------------------------------------
threshold: 0.35000000000000003
precision: 0.5899
recall: 0.6105
f1: 0.6000
--------------------------------------------------------------
threshold: 0.4
precision: 0.6267
recall: 0.5016
f1: 0.5572
-----------------------------------------------------

In [34]:
# 1) CatBoost 모델
cat_model = CatBoostClassifier(
    iterations=800,          # n_estimators 대응
    learning_rate=0.05,
    depth=6,                 # max_depth 대응 (보통 4~10)
    loss_function="Logloss", # 이진 분류
    random_seed=7777,
    verbose=0                # 학습 로그 끄기 (원하면 100 같은 값으로)
)

# 2) 학습
cat_model.fit(X_train, y_train)

# 3) 확률 예측 (양성=1 확률)
cat_prob = cat_model.predict_proba(X_test)[:, 1]

# 4) 임계값 스윕
thresholds = np.arange(0.05, 0.51, 0.05)

for t in thresholds:
    y_pred = (cat_prob >= t).astype(int)

    print(f"threshold: {t:.2f}")
    print(f"precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
    print(f"f1: {f1_score(y_test, y_pred, zero_division=0):.4f}")
    print("--------------------------------------------------------------")


threshold: 0.05
precision: 0.4369
recall: 0.9987
f1: 0.6079
--------------------------------------------------------------
threshold: 0.10
precision: 0.4555
recall: 0.9834
f1: 0.6226
--------------------------------------------------------------
threshold: 0.15
precision: 0.4741
recall: 0.9522
f1: 0.6330
--------------------------------------------------------------
threshold: 0.20
precision: 0.4949
recall: 0.9079
f1: 0.6406
--------------------------------------------------------------
threshold: 0.25
precision: 0.5239
recall: 0.8273
f1: 0.6415
--------------------------------------------------------------
threshold: 0.30
precision: 0.5581
recall: 0.7194
f1: 0.6286
--------------------------------------------------------------
threshold: 0.35
precision: 0.5894
recall: 0.6147
f1: 0.6018
--------------------------------------------------------------
threshold: 0.40
precision: 0.6266
recall: 0.5029
f1: 0.5580
--------------------------------------------------------------
threshold: 0.45
